# Clinical validation — SHaRe HCM WGS sub-cohort

**Goal 1 (this notebook, first pass):** reproduce the core genotype-outcome finding of Ho et al. 2018 (Circulation) — SARC+ patients have the worst outcomes, SARC− the best, SARC VUS intermediate — in this newer WGS-sequenced SHaRe sub-cohort (n=3,369), using the **DCC's own pre-existing genetic classification** (`HCM_DCC_SarcStatus`) as a benchmark. This deliberately does *not* yet use the CardioClassifier pipeline or the `priority` variant-annotation file — that comparison is deferred pending a decision on whether those precomputed annotations can be used as-is or should be recomputed live.

**Cohort caveat:** this is a WGS-sequenced subset of SHaRe, not the exact cohort/snapshot from the 2018 paper (which used site-level clinical genetic testing through Dec 2016; this is a later, WGS-based extract). Group sizes and the exact 9-gene sarcomere definition (`ACTC1, MYBPC3, MYH7, MYL2, MYL3, TNNC1, TNNI3, TNNT2, TPM1` — the DCC's own `PLP_<gene>`/`VUS_<gene>` columns) differ slightly from the paper's original 8-gene definition.

**Outcome:** "Overall composite" (`event_Composite_Overall_NoLVEF` / `t2_Composite_Overall`) — the closest match to the paper's Figure 3A end point (first occurrence of ventricular arrhythmia, HF, death, AF, or stroke).

**Time scale — from birth, not from enrollment.** `t2_*` columns are confirmed (checked against `age_Death`) to be **age**. Initially this notebook fit a left-truncated KM with entry = `FirstEncounter_Age`, assuming a prospective cohort where nothing is known before enrollment. That was wrong: SHaRe does retrospective chart review to ascertain events *before* a patient's first visit (stated explicitly in the paper's Methods), so a patient's outcome age can legitimately be younger than their first-encounter age — which is exactly why Figure 1's axis is "cumulative incidence **from birth**." Left-truncating at `FirstEncounter_Age` silently dropped 21% of the cohort (disproportionately the early-event patients) and produced nonsensical results (SARC+ showing *lower* risk than SARC−). Fixed by treating every patient as observed from age 0 — a standard (non-truncated) KM/Cox on age-at-event/censoring — matching the registry's own retrospective-ascertainment convention.

> **Restricted data — not distributed with this repository.**
>
> This notebook analyses individual-level clinical and genetic data from the
> Sarcomeric Human Cardiomyopathy Registry (SHaRe). Under SHaRe's data-use
> terms that data cannot be redistributed, so the input file paths below are
> placeholders (`<...>`) and every rendered cell output has been cleared.
>
> To run this notebook you need your own SHaRe data access. Point each
> placeholder path at your local copy of the corresponding table:
>
> | placeholder | contents |
> |---|---|
> | DCC clinical + survival table | one row per patient: `HCM_DCC_SarcStatus`, `HCM_DCC_GeneticStatus`, `t2_Composite_Overall`, `event_Composite_Overall_NoLVEF`, ages |
> | pipeline-classified variant table | one row per variant: `variant_id`, `gene_symbol`, `classification`, `vus_subtier` (produced by running the CardioClassifier pipeline over the cohort's variants) |
> | per-patient variant-carriage table | long `entity_sample_id` × `VariantID` table (SHaRe merged VCF) |
> | priority variant table | authoritative HCM gene whitelist via its `SYMBOL` column |
>
> Summary statistics quoted in the markdown below (hazard ratios, group
> sizes, risk percentages) are aggregate, non-identifiable results retained
> for methodology; redact them here too if your write-up requires it.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter, CoxPHFitter

pd.set_option("display.max_columns", 50)

DCC_PATH = "<SHaRe DCC clinical + survival-outcome table>"

In [ ]:
dcc = pd.read_csv(DCC_PATH, low_memory=False)
print(dcc.shape)
dcc["HCM_DCC_SarcStatus"].value_counts(dropna=False)

## Define SARC+/SARC−/SARC VUS groups

Mirrors the paper's own exclusions: patients with no genetic result (`SARC(0)`) and patients whose positive result is in a non-sarcomere/phenocopy context (`SARC(Genocopy)`, `SARC(Complex)`, `SARC(Non-Syndromic)`, `SARC(*)` — the DCC's own "not core sarcomere HCM" buckets) are dropped, the same way Ho et al. excluded GLA/LAMP2 phenocopy carriers. `SARC(2+)` (multiple P/LP sarcomere variants) is folded into `SARC+`, matching how the paper treats SARC2+ as a subset of SARC+.

In [ ]:
SARC_GROUP_MAP = {
    "SARC(+)": "SARC+",
    "SARC(2+)": "SARC+",
    "SARC(-)": "SARC-",
    "SARC(U)": "SARC VUS",
}

dcc["sarc_group"] = dcc["HCM_DCC_SarcStatus"].map(SARC_GROUP_MAP)

print("Excluded (no genetic result / non-core-sarcomere):")
print(dcc.loc[dcc["sarc_group"].isna(), "HCM_DCC_SarcStatus"].value_counts(dropna=False))
print()
print("Included groups:")
print(dcc["sarc_group"].value_counts(dropna=False))

In [ ]:
surv_cols = ["entity_sample_id", "sarc_group", "t2_Composite_Overall", "event_Composite_Overall_NoLVEF"]
surv = dcc[surv_cols].dropna(subset=["sarc_group", "t2_Composite_Overall", "event_Composite_Overall_NoLVEF"]).copy()
surv = surv.rename(columns={
    "t2_Composite_Overall": "exit",
    "event_Composite_Overall_NoLVEF": "event",
})

# exit is age at event/censoring, observed from birth (age 0) per SHaRe's retrospective
# ascertainment convention -- see markdown note above. Just needs to be positive.
bad = surv["exit"] <= 0
print(f"Dropping {bad.sum()} rows with non-positive exit age")
surv = surv[~bad]

print(f"Analysis n = {len(surv)} (of {dcc['sarc_group'].notna().sum()} in the 3 included groups)")
surv.groupby("sarc_group").agg(n=("event", "size"), events=("event", "sum"))

In [ ]:
# Colors mirror Figure 3A of Ho et al. 2018: SARC+ green, SARC- red, SARC VUS blue
# (kept as distinct paper-matching hues here, not the ordinal ramp used in Parts 2-3,
# since the point of this specific plot is visual side-by-side comparison with the
# published figure)
GROUP_COLORS = {"SARC+": "#2a9d55", "SARC-": "#d9534f", "SARC VUS": "#3b7dd8"}
GROUP_ORDER = ["SARC+", "SARC-", "SARC VUS"]

fig, ax = plt.subplots(figsize=(11, 8.5))

fitters = {}
for group in GROUP_ORDER:
    g = surv[surv["sarc_group"] == group]
    kmf = KaplanMeierFitter(label=f"{group} (n={len(g)})")
    kmf.fit(durations=g["exit"], event_observed=g["event"])  # observed from age 0 (birth)
    fitters[group] = kmf
    kmf.plot_survival_function(ax=ax, color=GROUP_COLORS[group], ci_show=True, ci_alpha=0.12, linewidth=2.2)

ax.set_xlim(0, 70)
ax.set_ylim(0, 1)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Proportion Free of Overall Composite Endpoint")
#ax.set_title("Freedom from overall composite outcome by sarcomere genotype\n(SHaRe WGS cohort, DCC-adjudicated genotype)")
ax.legend(loc="lower left", fontsize=11)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("outputs/km_overall_composite_by_sarcstatus.png", dpi=200)
plt.show()

## Significance testing

Now that this is a standard (non-truncated) survival setup, a plain pairwise log-rank test applies directly — no Cox workaround needed. Reporting both the log-rank p-value and a Cox-based hazard ratio (with 95% CI) per pair, mirroring how Figure 3/Table 2 of the paper report log-rank p-values alongside Cox HRs.

In [ ]:
from itertools import combinations
from lifelines.statistics import logrank_test

def pairwise_comparison(surv, group_a, group_b):
    pair = surv[surv["sarc_group"].isin([group_a, group_b])].copy()
    pair["group_bin"] = (pair["sarc_group"] == group_a).astype(int)

    a = pair[pair["group_bin"] == 1]
    b = pair[pair["group_bin"] == 0]
    lr = logrank_test(a["exit"], b["exit"], event_observed_A=a["event"], event_observed_B=b["event"])

    cph = CoxPHFitter()
    cph.fit(pair[["exit", "event", "group_bin"]], duration_col="exit", event_col="event")
    hr = np.exp(cph.params_["group_bin"])
    ci_lo, ci_hi = np.exp(cph.confidence_intervals_.loc["group_bin"])
    return hr, ci_lo, ci_hi, lr.p_value

print("Pairwise comparisons (Overall composite), HR relative to 2nd-listed group:\n")
for a, b in combinations(GROUP_ORDER, 2):
    hr, lo, hi, p = pairwise_comparison(surv, a, b)
    print(f"{a} vs {b}: HR={hr:.2f} (95% CI {lo:.2f}-{hi:.2f}), log-rank p={p:.4g}")

In [ ]:
# Cross-check against the paper's own headline number: "risk of overall composite by age 50
# is 29.1% for SARC+ vs 24.9% for SARC VUS vs 14.2% for SARC-" (Figure 3A legend)
print("Risk of overall composite by age 50 (this cohort vs. Ho et al. 2018):\n")
paper_risk_at_50 = {"SARC+": 0.291, "SARC VUS": 0.249, "SARC-": 0.142}
for group in GROUP_ORDER:
    surv_prob_50 = fitters[group].predict(50)
    this_risk = 1 - surv_prob_50
    print(f"{group}: this cohort = {this_risk:.1%}, paper = {paper_risk_at_50[group]:.1%}")

# Part 2 — pipeline-derived classification (CardioClassifier, live API calls)

Same question, but instead of trusting the DCC's own `HCM_DCC_SarcStatus` label, classification is now derived independently by CardioClassifier's own evidence-rule logic, run on live-fetched annotation (VEP consequence/HGVS + REVEL/CADD/AlphaMissense/SpliceAI, gnomAD v4 joint AC/AN/FAF95). **The priority table's precomputed annotations are not used as classification inputs** — its `VariantID` list is used only to identify which patients carry which variants (the join below); none of its annotation *values* feed into classification. Every score here comes from a live API call rather than that file.

**Known ceiling on this run:** curator-only evidence codes (PS2 de novo, PP1/BS4 segregation, PM3/BP2 phasing, PP4 phenotype-specificity, BS2 healthy-adult-observed) are not computed here — no per-patient clinical chart data exists at this batch/variant-only scale. Some variants a human curator would call Pathogenic (using segregation/de novo evidence) may therefore land as Likely Pathogenic or VUS-high here instead.

**Scope:**
- Restricted to the same 9-gene sarcomere definition (`ACTC1, MYBPC3, MYH7, MYL2, MYL3, TNNC1, TNNI3, TNNT2, TPM1`) — a variant classified Pathogenic in a non-sarcomere gene (e.g. a RASopathy phenocopy gene elsewhere in the 29-gene panel) does not count toward this grouping.
- **No separate rarity filter is applied on top of the classifier's output.** the per-patient variant table and the priority table are used *only* to identify which patients carry which variants — not as an input to classification itself, and not as a second population-frequency gate. The pipeline's own `apply_population_rules` already scores PM2/BA1/BS1 using gene-specific thresholds, so a variant common enough to matter is already pulled toward Benign/Likely Benign by the classifier itself; layering an extra flat AF cutoff on top would only risk wrongly discarding a variant the pipeline legitimately called VUS/LP.
- **VUS is split into its 3 subtiers** (VUS-high/mid/low, from `classifierr.ipynb`'s Tavtigian point-scale scorer) instead of one blended VUS group — testing whether VUS-high tracks closer to SARC+ and VUS-low closer to SARC−, which the DCC's single "SARC(U)" label can't distinguish.
- Per-patient group assignment uses a severity hierarchy (worst qualifying variant wins) when a patient carries more than one qualifying variant: Pathogenic/Likely Pathogenic > VUS-high > VUS-mid > VUS-low > none (→ pipeline SARC−). A patient whose only variants are Likely Benign/Benign is treated the same as a patient with no variant at all — both fall to SARC−.

In [ ]:
SARCOMERE_GENES = ["ACTC1", "MYBPC3", "MYH7", "MYL2", "MYL3", "TNNC1", "TNNI3", "TNNT2", "TPM1"]
LIVE_CLASSIFIED_PATH = "<pipeline-classified SHaRe variant table (from the CardioClassifier pipeline)>"
MERGED_VCF_PATH = "<per-patient variant-carriage table (SHaRe merged VCF)>"

live = pd.read_csv(LIVE_CLASSIFIED_PATH, low_memory=False)
print(f"Loaded {len(live)} classified variants")
print(live["gene_symbol"].isin(SARCOMERE_GENES).value_counts())
print()

sarc_live = live[live["gene_symbol"].isin(SARCOMERE_GENES)].copy()
print(f"Sarcomere-gene variants: {len(sarc_live)}")
sarc_live["classification"].value_counts(dropna=False)

In [ ]:
# By design, no separate ad hoc rarity gate is imposed on top of the
# classifier's own output. apply_population_rules already scores PM2/BA1/BS1 using
# gene/cspec-specific thresholds -- a variant common enough to matter is already
# pulled toward Benign/Likely Benign by the classifier itself. Adding our own flat
# AF<0.0001 cutoff on top would only risk wrongly excluding a variant the pipeline
# legitimately called VUS/LP (e.g. one sitting at AF=0.0003 that still clears a
# gene-specific PM2 threshold). So: use every sarcomere-gene variant's own
# `classification` directly, no extra filtering.
def to_tier(row):
    if row["classification"] in ("Pathogenic", "Likely Pathogenic"):
        return "SARC+"
    if row["classification"] == "VUS":
        return row["vus_subtier"]  # VUS-high / VUS-mid / VUS-low
    return None  # Likely Benign / Benign -- doesn't qualify as a "positive" finding

sarc_live["tier"] = sarc_live.apply(to_tier, axis=1)
print("Sarcomere-gene variants by tier (pipeline classification, no extra rarity gate):")
print(sarc_live["tier"].value_counts(dropna=False))

qualifying_variants = sarc_live.dropna(subset=["tier"])[["variant_id", "gene_symbol", "tier"]].copy()
print(f"\nTotal qualifying (sarcomere-gene, P/LP/VUS) variants: {len(qualifying_variants)}")

## Join qualifying variants back to patients

The per-patient variant table only ever contains rows where a variant was actually called present (confirmed: no `0/0` genotypes exist in this table at all — every row already means "this patient carries this variant"), so no genotype filtering is needed beyond the join itself.

In [ ]:
# Only pull the 3 columns needed, and only rows matching a qualifying variant --
# this table is large, so only the needed rows/columns are streamed in.
qualifying_ids = set(qualifying_variants["variant_id"])

carrier_chunks = []
for chunk in pd.read_csv(MERGED_VCF_PATH, sep="\t", usecols=["entity_sample_id", "VariantID"], chunksize=200_000):
    hit = chunk[chunk["VariantID"].isin(qualifying_ids)]
    if len(hit):
        carrier_chunks.append(hit)

carriers = pd.concat(carrier_chunks, ignore_index=True) if carrier_chunks else pd.DataFrame(columns=["entity_sample_id", "VariantID"])
carriers = carriers.merge(qualifying_variants, left_on="VariantID", right_on="variant_id", how="left")
print(f"Carrier (patient, qualifying variant) rows: {len(carriers)}")
print(f"Unique patients carrying >=1 qualifying variant: {carriers['entity_sample_id'].nunique()}")
carriers.head()

In [ ]:
# Worst-qualifying-variant-wins, same principle as SARC2+ folding into SARC+ in Part 1
TIER_RANK = {"SARC+": 4, "VUS-high": 3, "VUS-mid": 2, "VUS-low": 1}

carriers["tier_rank"] = carriers["tier"].map(TIER_RANK)
best_per_patient = carriers.sort_values("tier_rank", ascending=False).drop_duplicates("entity_sample_id", keep="first")
patient_tier = best_per_patient.set_index("entity_sample_id")["tier"]

print("Patients by worst-qualifying-tier (pipeline-derived):")
print(patient_tier.value_counts())

## Apply to the full cohort, not just the DCC-genotyped subset

Unlike Part 1 (limited to the 2,494 patients the DCC had a `HCM_DCC_SarcStatus` result for), this pipeline classification comes from the WGS data directly, which exists for **all 3,369 patients** regardless of whether they separately had clinical genetic testing. So every patient gets a pipeline tier: their worst qualifying variant if they carry one, otherwise pipeline-`SARC-` by default (no rare P/LP/VUS sarcomere variant found in their WGS). This is worth flagging as a genuine point of difference, not just noise — some `SARC(0)` (DCC: untested) patients will get a real answer here that the DCC benchmark simply couldn't provide.

In [ ]:
dcc["pipeline_group"] = dcc["entity_sample_id"].map(patient_tier).fillna("SARC-")
print(dcc["pipeline_group"].value_counts())
print()
print("Cross-tab vs DCC's own label (rows=pipeline, cols=DCC):")
pd.crosstab(dcc["pipeline_group"], dcc["HCM_DCC_SarcStatus"], dropna=False)

In [ ]:
surv2_cols = ["entity_sample_id", "pipeline_group", "t2_Composite_Overall", "event_Composite_Overall_NoLVEF"]
surv2 = dcc[surv2_cols].dropna(subset=["t2_Composite_Overall", "event_Composite_Overall_NoLVEF"]).copy()
surv2 = surv2.rename(columns={"t2_Composite_Overall": "exit", "event_Composite_Overall_NoLVEF": "event"})

bad2 = surv2["exit"] <= 0
print(f"Dropping {bad2.sum()} rows with non-positive exit age")
surv2 = surv2[~bad2]

print(f"Analysis n = {len(surv2)} (of {len(dcc)} full cohort)")
surv2.groupby("pipeline_group").agg(n=("event", "size"), events=("event", "sum"))

In [ ]:
# Fixed color convention across the whole notebook: SARC+ always green, SARC- always
# red, SARC VUS always blue (matching Figure 3A / Part 1). Where VUS is split into
# subtiers, all three stay in the blue family (darker = more severe) rather than
# introducing new hues, so "blue = uncertain" reads consistently everywhere.
GROUP2_COLORS = {
    "SARC+": "#2a9d55",
    "VUS-high": "#1c5cab",
    "VUS-mid": "#3b7dd8",
    "VUS-low": "#8fb8ea",
    "SARC-": "#d9534f",
}
GROUP2_ORDER = ["SARC+", "VUS-high", "VUS-mid", "VUS-low", "SARC-"]

fig, ax = plt.subplots(figsize=(11, 8.5))

fitters2 = {}
for group in GROUP2_ORDER:
    g = surv2[surv2["pipeline_group"] == group]
    if len(g) == 0:
        continue
    kmf = KaplanMeierFitter(label=f"{group} (n={len(g)})")
    kmf.fit(durations=g["exit"], event_observed=g["event"])
    fitters2[group] = kmf
    kmf.plot_survival_function(ax=ax, color=GROUP2_COLORS[group], ci_show=True, ci_alpha=0.12, linewidth=2.2)

ax.set_xlim(0, 70)
ax.set_ylim(0, 1)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Proportion Free of Overall Composite Endpoint")
ax.set_title("Freedom from overall composite outcome by genotype\n(SHaRe WGS cohort, reimplemented CardioClassifier classification, n=3,369)")
ax.legend(loc="lower left", fontsize=11)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("outputs/km_overall_composite_by_pipeline_tier.png", dpi=200)
plt.show()

In [ ]:
def pairwise_comparison2(surv, group_a, group_b):
    pair = surv[surv["pipeline_group"].isin([group_a, group_b])].copy()
    pair["group_bin"] = (pair["pipeline_group"] == group_a).astype(int)

    a = pair[pair["group_bin"] == 1]
    b = pair[pair["group_bin"] == 0]
    lr = logrank_test(a["exit"], b["exit"], event_observed_A=a["event"], event_observed_B=b["event"])

    cph = CoxPHFitter()
    cph.fit(pair[["exit", "event", "group_bin"]], duration_col="exit", event_col="event")
    hr = np.exp(cph.params_["group_bin"])
    ci_lo, ci_hi = np.exp(cph.confidence_intervals_.loc["group_bin"])
    return hr, ci_lo, ci_hi, lr.p_value

print("Each tier vs pipeline SARC-, HR relative to SARC-:\n")
for group in ["SARC+", "VUS-high", "VUS-mid", "VUS-low"]:
    if group not in fitters2:
        continue
    hr, lo, hi, p = pairwise_comparison2(surv2, group, "SARC-")
    print(f"{group} vs SARC-: HR={hr:.2f} (95% CI {lo:.2f}-{hi:.2f}), log-rank p={p:.4g}")

print("\nSARC+ vs VUS-high (does the top VUS subtier look like SARC+?):")
hr, lo, hi, p = pairwise_comparison2(surv2, "SARC+", "VUS-high")
print(f"HR={hr:.2f} (95% CI {lo:.2f}-{hi:.2f}), log-rank p={p:.4g}")

## Sensitivity analysis (Supplementary Figure): headline comparison through the Ho et al. (2018) covariate set

The main-text KM curves and hazard ratios are univariable (genotype group only) with attained age as the timescale — matching the *crude* genotype comparison in Ho et al. 2018 (Figure 3). Ho et al.'s multivariable model (their Table 2) additionally adjusted for **proband status, sex, and race**; a further model added **age at diagnosis** as a covariate (reported there as "data not shown"), and the headline comparison used **family-specific frailty** to handle relatedness.

This cell refits the headline **SARC+ vs SARC−** comparison (pipeline-derived groups) through that same sequence — crude → + proband/sex/race → + age at diagnosis — on one complete-case sample, with **`FamilyID`-clustered robust standard errors** as the `lifelines` stand-in for family frailty. It prints the model sequence and saves `outputs/hcm_sensitivity_hoetal_covariates_forest.png` for use as a supplementary figure/table. No main-text figure changes; the VUS-subtier and all-29-gene models remain univariable and the residual-confounding caveat still applies to those.


In [ ]:
# --------------------------------------------------
# Sensitivity analysis: SARC+ vs SARC- through the Ho et al. (2018) covariate set
# (Supplementary Figure/Table -- main-text KM/forest figures are unchanged)
# --------------------------------------------------
_cov = dcc[["entity_sample_id", "FamilyID", "Sex", "Race",
            "HCM_DCC_IsProband", "Primary_Diagnosis_Age"]].copy()
_d = (surv2[surv2["pipeline_group"].isin(["SARC+", "SARC-"])]
      .merge(_cov, on="entity_sample_id", how="left"))

_d["group_bin"]  = (_d["pipeline_group"] == "SARC+").astype(int)                      # 1 = SARC+
_sex = _d["Sex"].astype(str).str.strip().str.lower()
_d["sex_female"] = _sex.map({"female": 1, "f": 1, "male": 0, "m": 0})                 # 1 = female
_race = _d["Race"].astype(str).str.strip().str.lower()
_d["nonwhite"]   = np.where(_d["Race"].isna(), np.nan, (_race != "white").astype(float))  # 1 = non-white
_d["proband"]    = pd.to_numeric(_d["HCM_DCC_IsProband"], errors="coerce")            # 1 = proband
_d["age_at_dx"]  = pd.to_numeric(_d["Primary_Diagnosis_Age"], errors="coerce")        # years

_needed = ["exit", "event", "group_bin", "proband", "sex_female", "nonwhite", "age_at_dx", "FamilyID"]
_cc = _d.dropna(subset=_needed).copy()
_cc["FamilyID"] = _cc["FamilyID"].astype(str)
_n_drop = len(_d) - len(_cc)
if _d["sex_female"].isna().all():
    raise ValueError(f"Sex column not recognised -- observed: {sorted(_d['Sex'].dropna().astype(str).unique())[:6]}")

def _fit(model_cols):
    """Cox fit with FamilyID-clustered robust SEs; falls back to unclustered on old lifelines."""
    try:
        m = CoxPHFitter().fit(_cc[model_cols + ["FamilyID"]], duration_col="exit",
                              event_col="event", cluster_col="FamilyID")
        return m, "FamilyID-clustered robust SE"
    except (TypeError, KeyError, ValueError):
        m = CoxPHFitter().fit(_cc[model_cols], duration_col="exit", event_col="event")
        return m, "unclustered SE (cluster_col unavailable)"

_M1, _se = _fit(["exit", "event", "group_bin"])
_M2, _   = _fit(["exit", "event", "group_bin", "proband", "sex_female", "nonwhite"])
_M3, _   = _fit(["exit", "event", "group_bin", "proband", "sex_female", "nonwhite", "age_at_dx"])

def _hr(m, term):
    lo, hi = np.exp(m.confidence_intervals_.loc[term].values)
    return np.exp(m.params_[term]), lo, hi, m.summary.loc[term, "p"]

print(f"SARC+ vs SARC- (pipeline-derived).  Complete-case n = {len(_cc)} "
      f"({_n_drop} dropped for missing covariates);  {_se}.\n")
_g1, _g2, _g3 = _hr(_M1, "group_bin"), _hr(_M2, "group_bin"), _hr(_M3, "group_bin")
print(f"  M1  crude (genotype only)                       HR {_g1[0]:.2f} ({_g1[1]:.2f}-{_g1[2]:.2f}), p={_g1[3]:.2g}")
print(f"  M2  + proband status + sex + race (Ho Table 2)   HR {_g2[0]:.2f} ({_g2[1]:.2f}-{_g2[2]:.2f}), p={_g2[3]:.2g}")
print(f"  M3  + age at diagnosis                           HR {_g3[0]:.2f} ({_g3[1]:.2f}-{_g3[2]:.2f}), p={_g3[3]:.2g}")
print("\n  M3 covariate hazard ratios:")
for _t, _lab in [("proband", "Proband vs relative"), ("sex_female", "Female vs male"),
                 ("nonwhite", "Non-white vs white"), ("age_at_dx", "Age at diagnosis, per year")]:
    _h = _hr(_M3, _t)
    print(f"    {_lab:26s} HR {_h[0]:.2f} ({_h[1]:.2f}-{_h[2]:.2f}), p={_h[3]:.2g}")

# ---- Supplementary forest plot ----
_rows = [
    ("SARC+ vs SARC-  (crude)",                    *_g1[:3], "#8a8a8a"),
    ("SARC+ vs SARC-  (+ proband, sex, race)",     *_g2[:3], "#5aa06e"),
    ("SARC+ vs SARC-  (+ age at diagnosis)",       *_g3[:3], "#2a9d55"),
    ("Female vs male",                             *_hr(_M3, "sex_female")[:3], "#3b7dd8"),
    ("Age at diagnosis, per year",                 *_hr(_M3, "age_at_dx")[:3],  "#3b7dd8"),
    ("Proband vs relative",                        *_hr(_M3, "proband")[:3],    "#9aa7b1"),
    ("Non-white vs white",                         *_hr(_M3, "nonwhite")[:3],   "#9aa7b1"),
]
fig, ax = plt.subplots(figsize=(9.5, 5))
_y = list(range(len(_rows), 0, -1))
for y, (label, hr, lo, hi, color) in zip(_y, _rows):
    ax.errorbar(hr, y, xerr=[[hr - lo], [hi - hr]], fmt="o", color=color, ecolor=color,
                elinewidth=2, capsize=4, markersize=9)
    ax.text(hi * 1.05, y, f"{hr:.2f} ({lo:.2f}-{hi:.2f})", va="center", fontsize=10, color="#333333")
ax.axvline(1.0, color="#888888", linewidth=1.2, linestyle="--")
ax.set_xscale("log")
ax.set_xticks([0.5, 1, 1.5, 2, 3])
ax.get_xaxis().set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())
ax.set_yticks(_y)
ax.set_yticklabels([r[0] for r in _rows], fontsize=10)
ax.set_ylim(0.4, len(_rows) + 0.6)
ax.set_xlabel("Hazard ratio (log scale); FamilyID-clustered 95% CI")
ax.set_title("Sensitivity analysis: SARC+ vs SARC- genotype effect through the\n"
             f"Ho et al. (2018) covariate set (pipeline-derived groups, complete-case n={len(_cc)})",
             fontsize=11)
ax.grid(axis="x", alpha=0.3)
for _s in ["top", "right", "left"]:
    ax.spines[_s].set_visible(False)
fig.tight_layout()
fig.savefig("outputs/hcm_sensitivity_hoetal_covariates_forest.png", dpi=300, facecolor="white")
plt.show()


## Decluttered views: VUS subtiers alone, and a forest plot

The 5-curve plot above is comprehensive but busy. Two more focused views of the same underlying `fitters2`/HR results: a KM plot with just the three VUS subtiers front and center (SARC+/SARC− shown only as thin gray reference lines, not competing for attention), and a forest plot of the HRs vs SARC− — the standard, compact way to show several point estimates + CIs at once (mirrors Figure 4's style in Ho et al. 2018) without needing overlapping survival curves at all.

In [ ]:
VUS_ONLY_COLORS = {"VUS-high": "#1c5cab", "VUS-mid": "#3b7dd8", "VUS-low": "#8fb8ea"}
VUS_ONLY_ORDER = ["VUS-high", "VUS-mid", "VUS-low"]

fig, ax = plt.subplots(figsize=(11, 8.5))

# SARC+/SARC- as thin dashed/dotted reference lines, still in their fixed
# green/red (not gray) so the color convention stays consistent across the notebook
for group, style in [("SARC+", "--"), ("SARC-", ":")]:
    fitters2[group].plot_survival_function(ax=ax, color=GROUP2_COLORS[group], ci_show=False, linewidth=1.6, linestyle=style)

for group in VUS_ONLY_ORDER:
    g = surv2[surv2["pipeline_group"] == group]
    kmf = fitters2[group]
    kmf.plot_survival_function(ax=ax, color=VUS_ONLY_COLORS[group], ci_show=True, ci_alpha=0.15, linewidth=2.4)

ax.set_xlim(0, 70)
ax.set_ylim(0, 1)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Proportion Free of Overall Composite Endpoint")
ax.set_title("Variants of uncertain significance, by subtier\n(SHaRe WGS cohort, reimplemented CardioClassifier classification;\nSARC+/SARC- shown as thin reference lines)")
ax.legend(loc="lower left", fontsize=11)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("outputs/km_vus_subtiers_only.png", dpi=200)
plt.show()

In [ ]:
forest_order = ["SARC+", "VUS-high", "VUS-mid", "VUS-low", "SARC-"]  # top to bottom, most to least severe
forest_rows = []
for group in forest_order[:-1]:
    hr, lo, hi, p = pairwise_comparison2(surv2, group, "SARC-")
    forest_rows.append({"group": group, "hr": hr, "lo": lo, "hi": hi, "p": p, "is_ref": False})
# SARC- is the reference group -- HR=1.0 by definition, no CI of its own to plot.
# Shown explicitly as a diamond so the comparison target isn't left implicit.
n_ref = surv2[surv2["pipeline_group"] == "SARC-"].shape[0]
forest_rows.append({"group": "SARC-", "hr": 1.0, "lo": None, "hi": None, "p": None, "is_ref": True, "n": n_ref})

fig, ax = plt.subplots(figsize=(9, 6))
y_positions = list(range(len(forest_rows), 0, -1))  # top row = highest y

for y, row in zip(y_positions, forest_rows):
    color = GROUP2_COLORS[row["group"]]
    if row["is_ref"]:
        ax.plot(1.0, y, marker="D", color=color, markersize=10, markeredgecolor="#333333", markeredgewidth=0.8)
        ax.text(1.35, y, f"1.00 (reference), n={row['n']}", va="center", fontsize=10, color="#333333")
    else:
        ax.errorbar(
            row["hr"], y,
            xerr=[[row["hr"] - row["lo"]], [row["hi"] - row["hr"]]],
            fmt="o", color=color, ecolor=color, elinewidth=2, capsize=4, markersize=9,
        )
        label = f"{row['hr']:.2f} ({row['lo']:.2f}-{row['hi']:.2f}), p={row['p']:.2g}"
        ax.text(row["hi"] * 1.08, y, label, va="center", fontsize=10, color="#333333")

ax.axvline(1.0, color="#888888", linewidth=1.2, linestyle="--")
ax.set_xscale("log")
ax.set_xticks([0.5, 1, 1.5, 2, 3])
ax.get_xaxis().set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())
ax.set_yticks(y_positions)
ax.set_yticklabels(forest_order, fontsize=11)
ax.set_xlim(0.6, 3.5)
ax.set_ylim(0.4, len(forest_rows) + 0.6)
ax.set_xlabel("Hazard ratio vs SARC- (log scale)")
ax.set_title("Overall composite outcome by genotype\n(SHaRe WGS cohort, reimplemented CardioClassifier classification;\ndashed line = no effect, diamond = reference group)")
ax.grid(axis="x", alpha=0.3)
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
fig.tight_layout()
fig.savefig("outputs/forest_plot_vus_tiers.png", dpi=200)
plt.show()

In [ ]:
print("Risk of overall composite by age 50, pipeline-derived tiers:\n")
for group in GROUP2_ORDER:
    if group not in fitters2:
        continue
    this_risk = 1 - fitters2[group].predict(50)
    print(f"{group}: {this_risk:.1%} (n={surv2[surv2['pipeline_group']==group].shape[0]})")

# Part 1 vs Part 2 — summary

**Methodology note:** an earlier version of Part 2 applied an extra ad hoc rarity filter (gnomAD AF<0.0001) on top of the classifier's own output; this was later removed, on the principle that the per-patient variant table and the priority table should only identify which patients carry which variants, not act as a second input to classification. With the gate removed, every sarcomere-gene variant's own pipeline `classification` is now used directly (population frequency is already handled internally by `apply_population_rules`'s PM2/BA1/BS1 logic). The numbers below reflect the corrected methodology; they moved only slightly from the rarity-gated version (e.g. SARC+ vs SARC− HR 1.88→1.87), which is itself a reassuring robustness check — the earlier ad hoc filter wasn't secretly driving the result.

| | DCC benchmark (Part 1) | CardioClassifier pipeline (Part 2) |
|---|---|---|
| SARC+ vs SARC− HR | 1.91 (1.68–2.17) | 1.87 (1.65–2.12) |
| Risk by age 50, SARC+ | 28.8% | 27.1% |
| Risk by age 50, SARC− | 10.5% | 9.3% |

The top-line SARC+/SARC− signal is nearly identical between the DCC's clinically-adjudicated labels and CardioClassifier's fully automated, live-API-derived classification — despite the pipeline having no access to segregation, de novo, phasing, or phenotype-specificity evidence. That's a meaningful validation result on its own: an automated ACMG pipeline with a known evidentiary ceiling still recovers essentially the same effect size a human curation team found.

**The more novel result is the VUS subtiering**, which the DCC's single blended `SARC(U)` label can't show:

| Tier | HR vs SARC- | Risk by age 50 |
|---|---|---|
| SARC+ | 1.87 (1.65–2.12) | 27.1% |
| VUS-high | 1.66 (1.35–2.04) | 25.5% |
| VUS-mid | 1.19 (0.98–1.45), border-significant | 15.5% |
| VUS-low | 0.98 (0.84–1.15), not significant | 9.8% |
| SARC- | (reference) | 9.3% |

**VUS-high is statistically indistinguishable from SARC+** (HR=1.14, p=0.21) and clinically behaves like it (25.5% vs 27.1% risk by 50) — consistent with the idea that most VUS-high variants are, in reality, pathogenic and simply haven't been reclassified yet. **VUS-low is statistically indistinguishable from SARC−** (HR=0.98, p=0.83) — consistent with most VUS-low variants being functionally benign. Ho et al.'s single "SARC VUS = intermediate risk" finding turns out to be an average over two quite different populations that the point-scale subtiering separates cleanly.

**Caveats to carry into the write-up:**
- This pipeline run has a known ceiling (no curator-only evidence: de novo, segregation, phasing, phenotype-specificity) — some true Pathogenic calls likely sit in Likely Pathogenic/VUS-high here instead.
- Part 2 covers the full WGS cohort (n=3,369), not just the DCC-genotyped subset (n=2,494) Part 1 used — a genuine advantage of WGS-based classification, but means the two n's aren't identical.
- REVEL/CADD/AlphaMissense/SpliceAI were fetched live rather than read from the priority table's precomputed annotations — cross-checked against that file for one variant anyway out of curiosity and found identical values, but the live fetch is what's actually driving every number in this notebook, not the file.

# Part 3 — all 29 HCM genes, not just the 9 sarcomere genes

Parts 1-2 deliberately restricted to the 9 core sarcomere genes, to make a clean apples-to-apples replication of Ho et al. Part 3 asks the broader question instead: does *any* P/LP/VUS variant across the full HCM gene panel (sarcomere + phenocopy/genocopy genes like `GLA`, `LAMP2`, `PRKAG2`, `TTR`, plus RASopathy genes `PTPN11`/`RAF1`/`RIT1`, plus other cardiomyopathy genes `FLNC`, `PLN`, `DES`, etc.) associate with worse outcome — benchmarked against the DCC's own **`HCM_DCC_GeneticStatus`** field (G(+)/G(-)/G(U)), which is the DCC's all-gene analog of `HCM_DCC_SarcStatus`.

**Gene list caveat (important):** the live pipeline's raw `gene_symbol` field has **50** unique values, not 29 — VEP sometimes annotates a variant against a nearby overlapping gene (antisense RNAs like `TPM1-AS`, a pseudogene, mitochondrial genes, even an unrelated gene like `SPI1`) rather than the intended panel gene, when a variant's genomic position overlaps multiple transcripts. Using the raw 50 values would silently pull in noise. The priority table's `SYMBOL` column gives the authoritative panel gene list (28 genes found there, curated to the actual target regions) — used here as the whitelist instead.

**Same as Part 2:** no separate rarity gate on top of the classifier's output — every variant's own `classification` is used directly, with the per-patient variant table and the priority table serving only to identify which patients carry which variants.

In [ ]:
# Authoritative 29-gene panel whitelist, sourced from the priority table's SYMBOL column
# (NOT the raw live-pipeline gene_symbol, which picked up 50 values incl. off-target genes)
priority_genes = pd.read_csv("<SHaRe priority variant table (authoritative HCM gene whitelist)>", usecols=["SYMBOL"])
ALL_HCM_GENES = sorted(priority_genes["SYMBOL"].dropna().unique().tolist())
print(f"Authoritative panel: {len(ALL_HCM_GENES)} genes")
print(ALL_HCM_GENES)

off_target = set(live["gene_symbol"].dropna().unique()) - set(ALL_HCM_GENES)
print(f"\nOff-target gene_symbol values excluded ({len(off_target)}): {sorted(off_target)}")

In [ ]:
all_gene_live = live[live["gene_symbol"].isin(ALL_HCM_GENES)].copy()
print(f"All-29-gene variants: {len(all_gene_live)} (vs {len(sarc_live)} sarcomere-only)")

# Same fix as Part 2: no separate rarity gate, use the pipeline's own classification directly.
all_gene_live["tier"] = all_gene_live.apply(to_tier, axis=1)
print()
print("All-29-gene variants by tier (pipeline classification, no extra rarity gate):")
print(all_gene_live["tier"].value_counts(dropna=False))

qualifying_variants_all = all_gene_live.dropna(subset=["tier"])[["variant_id", "gene_symbol", "tier"]].copy()
print(f"\nTotal qualifying (any of 29 genes, P/LP/VUS) variants: {len(qualifying_variants_all)}")

In [ ]:
qualifying_ids_all = set(qualifying_variants_all["variant_id"])

carrier_chunks_all = []
for chunk in pd.read_csv(MERGED_VCF_PATH, sep="\t", usecols=["entity_sample_id", "VariantID"], chunksize=200_000):
    hit = chunk[chunk["VariantID"].isin(qualifying_ids_all)]
    if len(hit):
        carrier_chunks_all.append(hit)

carriers_all = pd.concat(carrier_chunks_all, ignore_index=True) if carrier_chunks_all else pd.DataFrame(columns=["entity_sample_id", "VariantID"])
carriers_all = carriers_all.merge(qualifying_variants_all, left_on="VariantID", right_on="variant_id", how="left")
print(f"Carrier (patient, qualifying variant) rows: {len(carriers_all)}")
print(f"Unique patients carrying >=1 qualifying variant (any of 29 genes): {carriers_all['entity_sample_id'].nunique()}")

carriers_all["tier_rank"] = carriers_all["tier"].map(TIER_RANK)
best_per_patient_all = carriers_all.sort_values("tier_rank", ascending=False).drop_duplicates("entity_sample_id", keep="first")
patient_tier_all = best_per_patient_all.set_index("entity_sample_id")["tier"]

# Relabel SARC+ -> G+ for this all-29-gene scope, to avoid confusion with Part 2's
# sarcomere-specific SARC+ groups when both appear in the same notebook.
RELABEL_ALL_GENES = {"SARC+": "G+", "VUS-high": "GVUS-high", "VUS-mid": "GVUS-mid", "VUS-low": "GVUS-low"}
dcc["pipeline_group_all"] = dcc["entity_sample_id"].map(patient_tier_all).map(RELABEL_ALL_GENES).fillna("G-")
print()
print("Patients by worst-qualifying-tier (pipeline-derived, all 29 genes):")
print(dcc["pipeline_group_all"].value_counts())

In [ ]:
GALL_DCC_MAP = {"G(+)": "G+", "G(-)": "G-", "G(U)": "GVUS"}
dcc["dcc_gall_group"] = dcc["HCM_DCC_GeneticStatus"].map(GALL_DCC_MAP)

surv3_dcc = dcc[["entity_sample_id", "dcc_gall_group", "t2_Composite_Overall", "event_Composite_Overall_NoLVEF"]].dropna(
    subset=["dcc_gall_group", "t2_Composite_Overall", "event_Composite_Overall_NoLVEF"]
).copy()
surv3_dcc = surv3_dcc.rename(columns={"t2_Composite_Overall": "exit", "event_Composite_Overall_NoLVEF": "event"})
surv3_dcc = surv3_dcc[surv3_dcc["exit"] > 0]

print(f"DCC G+/G-/GVUS benchmark, n = {len(surv3_dcc)}")
surv3_dcc.groupby("dcc_gall_group").agg(n=("event", "size"), events=("event", "sum"))

In [ ]:
def fit_km_groups(surv_df, group_col, order, colors):
    fig, ax = plt.subplots(figsize=(11, 8.5))
    fitters = {}
    for group in order:
        g = surv_df[surv_df[group_col] == group]
        if len(g) == 0:
            continue
        kmf = KaplanMeierFitter(label=f"{group} (n={len(g)})")
        kmf.fit(durations=g["exit"], event_observed=g["event"])
        fitters[group] = kmf
        kmf.plot_survival_function(ax=ax, color=colors[group], ci_show=True, ci_alpha=0.12, linewidth=2.2)
    ax.set_xlim(0, 70)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Age (years)")
    ax.set_ylabel("Proportion Free of Overall Composite Endpoint")
    ax.legend(loc="lower left", fontsize=11)
    ax.grid(alpha=0.3)
    return fig, ax, fitters

def pairwise_generic(surv_df, group_col, group_a, group_b):
    pair = surv_df[surv_df[group_col].isin([group_a, group_b])].copy()
    pair["group_bin"] = (pair[group_col] == group_a).astype(int)
    a, b = pair[pair["group_bin"] == 1], pair[pair["group_bin"] == 0]
    lr = logrank_test(a["exit"], b["exit"], event_observed_A=a["event"], event_observed_B=b["event"])
    cph = CoxPHFitter()
    cph.fit(pair[["exit", "event", "group_bin"]], duration_col="exit", event_col="event")
    hr = np.exp(cph.params_["group_bin"])
    ci_lo, ci_hi = np.exp(cph.confidence_intervals_.loc["group_bin"])
    return hr, ci_lo, ci_hi, lr.p_value

# DCC's own genetic-status benchmark, all 29 genes. Same fixed convention as
# everywhere else: positive=green, negative=red, uncertain=blue.
DCC_GALL_COLORS = {"G+": "#2a9d55", "GVUS": "#3b7dd8", "G-": "#d9534f"}
fig, ax, fitters3a = fit_km_groups(surv3_dcc, "dcc_gall_group", ["G+", "GVUS", "G-"], DCC_GALL_COLORS)
ax.set_title("Freedom from overall composite outcome by genetic status\n(SHaRe WGS cohort, DCC-adjudicated genotype, all 29 genes, n=%s)" % f"{len(surv3_dcc):,}")
fig.tight_layout()
fig.savefig("outputs/km_overall_composite_by_dcc_geneticstatus.png", dpi=200)
plt.show()

print("\nDCC benchmark pairwise comparisons:")
for a, b in combinations(["G+", "G-", "GVUS"], 2):
    hr, lo, hi, p = pairwise_generic(surv3_dcc, "dcc_gall_group", a, b)
    print(f"{a} vs {b}: HR={hr:.2f} (95% CI {lo:.2f}-{hi:.2f}), log-rank p={p:.4g}")

In [ ]:
# pipeline-derived, all 29 genes, full cohort
surv3 = dcc[["entity_sample_id", "pipeline_group_all", "t2_Composite_Overall", "event_Composite_Overall_NoLVEF"]].dropna(
    subset=["t2_Composite_Overall", "event_Composite_Overall_NoLVEF"]
).copy()
surv3 = surv3.rename(columns={"t2_Composite_Overall": "exit", "event_Composite_Overall_NoLVEF": "event"})
surv3 = surv3[surv3["exit"] > 0]

GALL_ORDER = ["G+", "GVUS-high", "GVUS-mid", "GVUS-low", "G-"]
GALL_COLORS = {"G+": "#2a9d55", "GVUS-high": "#1c5cab", "GVUS-mid": "#3b7dd8", "GVUS-low": "#8fb8ea", "G-": "#d9534f"}

fig, ax, fitters3b = fit_km_groups(surv3, "pipeline_group_all", GALL_ORDER, GALL_COLORS)
ax.set_title("Freedom from overall composite outcome by genetic status\n(SHaRe WGS cohort, reimplemented CardioClassifier classification, all 29 genes, n=%s)" % f"{len(surv3):,}")
fig.tight_layout()
fig.savefig("outputs/km_overall_composite_by_pipeline_all_genes.png", dpi=200)
plt.show()

print("\nPipeline all-29-gene pairwise comparisons vs G-:")
for group in ["G+", "GVUS-high", "GVUS-mid", "GVUS-low"]:
    if group not in fitters3b:
        continue
    hr, lo, hi, p = pairwise_generic(surv3, "pipeline_group_all", group, "G-")
    print(f"{group} vs G-: HR={hr:.2f} (95% CI {lo:.2f}-{hi:.2f}), log-rank p={p:.4g}")

print("\nRisk of overall composite by age 50:")
for group in GALL_ORDER:
    if group not in fitters3b:
        continue
    risk = 1 - fitters3b[group].predict(50)
    n = surv3[surv3["pipeline_group_all"] == group].shape[0]
    print(f"{group}: {risk:.1%} (n={n})")

# Part 3 summary — does gene scope matter?

| | Part 1/2 (9 sarcomere genes) | Part 3 (all 29 genes) |
|---|---|---|
| DCC benchmark: positive vs negative HR | 1.91 (SARC+ vs SARC−) | 1.87 (G+ vs G−) |
| Pipeline: positive vs negative HR | 1.87 (SARC+ vs SARC−) | 1.89 (G+ vs G−) |
| Pipeline: VUS-low vs negative | HR=0.98, p=0.83 (indistinguishable) | HR=1.15, p=0.28 (still not significant, but noisier) |
| Pipeline: VUS-mid vs negative | HR=1.19, p=0.08 (borderline) | HR=1.26, p=0.10 (borderline) |
| "Negative" group size (pipeline) | n=1223 (36% of cohort) | n=139 (4% of cohort) |

**The P/LP signal (G+ vs G-) is essentially scope-independent** — restricting to sarcomere genes or using all 29 barely moves the headline HR (1.87-1.89 either way), and this held even after removing the ad hoc rarity gate (Part 2/3 were re-derived without that gate: no separate frequency filter on top of the classifier's own PM2/BA1/BS1 logic — see the methodology note above Part 1 vs Part 2). That's reassuring: the core finding isn't an artifact of gene selection or of the earlier filtering choice.

**But the "negative" reference group degrades even further once the net widens to 29 genes**, now that the extra rarity gate is gone. With every classified variant across 29 genes contributing (no external AF cutoff), pipeline-`G-` shrinks to an even smaller residual of 139-148 patients (~4% of the cohort) with an implausibly low ~5% risk by age 50 (vs. 9.3% for the properly-sized, ~1,200-patient sarcomere-restricted SARC− group) — too small and too skewed to trust as a stable reference, which is also why the VUS-subtier comparisons against it come out noisier (p=0.10-0.28 instead of Part 2's clean p=0.83/0.08/9e-7 gradient).

**Conclusion for the write-up:** the sarcomere-gene restriction in Parts 1-2 isn't just cosmetic fidelity to Ho et al.'s original definition — it's load-bearing for getting a well-powered, interpretable "negative" reference group. The all-29-gene version (Part 3) is worth reporting as a robustness check on the *positive* signal (which holds up), but Part 2's sarcomere-restricted, VUS-subtiered result is the one to lead with.

# Combined 4-panel figure v2 (publication-ready)

Restructured: the DCC-replication panel (not a main result, replicates
prior work) moves to supplement; the redundant all-5-solid-color KM panel is dropped in favour
of the reference-line style (same data, clearer VUS-subtier focus). Self-contained: reloads the DCC table,
the live classification, and the per-patient variant table directly from disk for both gene scopes.

- **A** -- KM curves, 9 core sarcomere genes (pipeline-derived), VUS split into 3 subtiers,
  SARC+/SARC- shown as thin dashed/dotted reference lines
- **B** -- forest plot of hazard ratios vs. SARC-, 9 core sarcomere genes, all four tiers plus
  the reference diamond
- **C** -- same as A, but for the all-HCM-gene panel (28 genes with qualifying variants in this
  cohort, sourced from the priority table's SYMBOL column -- confirm against the intended 29-gene
  target list before citing this count)
- **D** -- forest plot of hazard ratios vs. G-, same 28-gene panel


In [ ]:
# Combined 4-panel figure v2 (publication-ready) -- restructured:
# drop the DCC-replication panel (not a main result -> supplement) and the redundant
# all-5-solid-color KM panel (identical data to the reference-line panel, just restyled).
# New layout: A) KM, 9 core sarcomere genes (VUS subtiers + SARC+/SARC- reference lines)
#             B) HR forest plot, 9 core sarcomere genes
#             C) KM, all 29 HCM genes (same reference-line style as A)
#             D) HR forest plot, all 29 HCM genes
# Self-contained: reloads the DCC table, live classification, and per-patient variant table directly from disk.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

DCC_PATH = "<SHaRe DCC clinical + survival-outcome table>"
LIVE_CLASSIFIED_PATH = "<pipeline-classified SHaRe variant table (from the CardioClassifier pipeline)>"
MERGED_VCF_PATH = "<per-patient variant-carriage table (SHaRe merged VCF)>"
PRIORITY_PATH = "<SHaRe priority variant table (authoritative HCM gene whitelist)>"
SARCOMERE_GENES = ["ACTC1", "MYBPC3", "MYH7", "MYL2", "MYL3", "TNNC1", "TNNI3", "TNNT2", "TPM1"]  # 9 genes, incl. TNNC1 (2024 ClinGen upgrade)

print("Loading DCC...")
dcc = pd.read_csv(DCC_PATH, low_memory=False)

print("Loading live classification...")
live = pd.read_csv(LIVE_CLASSIFIED_PATH, low_memory=False)

def to_tier(row):
    if row["classification"] in ("Pathogenic", "Likely Pathogenic"):
        return "SARC+"
    if row["classification"] == "VUS":
        return row["vus_subtier"]
    return None

TIER_RANK = {"SARC+": 4, "VUS-high": 3, "VUS-mid": 2, "VUS-low": 1}

def build_patient_tier(gene_list, qualifying_ids_label):
    gene_live = live[live["gene_symbol"].isin(gene_list)].copy()
    gene_live["tier"] = gene_live.apply(to_tier, axis=1)
    qualifying_variants = gene_live.dropna(subset=["tier"])[["variant_id", "gene_symbol", "tier"]].copy()
    qualifying_ids = set(qualifying_variants["variant_id"])
    print(f"Scanning the per-patient variant table for {len(qualifying_ids)} qualifying variants ({qualifying_ids_label})...")
    carrier_chunks = []
    for chunk in pd.read_csv(MERGED_VCF_PATH, sep="\t", usecols=["entity_sample_id", "VariantID"], chunksize=200_000):
        hit = chunk[chunk["VariantID"].isin(qualifying_ids)]
        if len(hit):
            carrier_chunks.append(hit)
    carriers = pd.concat(carrier_chunks, ignore_index=True) if carrier_chunks else pd.DataFrame(columns=["entity_sample_id", "VariantID"])
    carriers = carriers.merge(qualifying_variants, left_on="VariantID", right_on="variant_id", how="left")
    carriers["tier_rank"] = carriers["tier"].map(TIER_RANK)
    best_per_patient = carriers.sort_values("tier_rank", ascending=False).drop_duplicates("entity_sample_id", keep="first")
    return best_per_patient.set_index("entity_sample_id")["tier"]

# ---- 9-gene (core sarcomere) pipeline-derived groups ----
patient_tier_sarc = build_patient_tier(SARCOMERE_GENES, "9 core sarcomere genes")
dcc["pipeline_group"] = dcc["entity_sample_id"].map(patient_tier_sarc).fillna("SARC-")
surv2 = dcc[["entity_sample_id", "pipeline_group", "t2_Composite_Overall", "event_Composite_Overall_NoLVEF"]].dropna(
    subset=["t2_Composite_Overall", "event_Composite_Overall_NoLVEF"]
).copy()
surv2 = surv2.rename(columns={"t2_Composite_Overall": "exit", "event_Composite_Overall_NoLVEF": "event"})
surv2 = surv2[surv2["exit"] > 0]

# ---- 29-gene (all HCM genes) pipeline-derived groups ----
priority_genes = pd.read_csv(PRIORITY_PATH, usecols=["SYMBOL"])
ALL_HCM_GENES = sorted(priority_genes["SYMBOL"].dropna().unique().tolist())
print(f"29-gene panel: {len(ALL_HCM_GENES)} genes")

patient_tier_all = build_patient_tier(ALL_HCM_GENES, "all 29 HCM genes")
RELABEL_ALL_GENES = {"SARC+": "G+", "VUS-high": "GVUS-high", "VUS-mid": "GVUS-mid", "VUS-low": "GVUS-low"}
dcc["pipeline_group_all"] = dcc["entity_sample_id"].map(patient_tier_all).map(RELABEL_ALL_GENES).fillna("G-")
surv3 = dcc[["entity_sample_id", "pipeline_group_all", "t2_Composite_Overall", "event_Composite_Overall_NoLVEF"]].dropna(
    subset=["t2_Composite_Overall", "event_Composite_Overall_NoLVEF"]
).copy()
surv3 = surv3.rename(columns={"t2_Composite_Overall": "exit", "event_Composite_Overall_NoLVEF": "event"})
surv3 = surv3[surv3["exit"] > 0]

print(f"surv2 (9-gene) n={len(surv2)}, surv3 (29-gene) n={len(surv3)}")

# ============================================================
# Plotting
# ============================================================
GROUP2_COLORS = {"SARC+": "#2a9d55", "VUS-high": "#1c5cab", "VUS-mid": "#3b7dd8", "VUS-low": "#8fb8ea", "SARC-": "#d9534f"}
GROUP2_ORDER = ["SARC+", "VUS-high", "VUS-mid", "VUS-low", "SARC-"]
VUS_ONLY_COLORS = {"VUS-high": "#1c5cab", "VUS-mid": "#3b7dd8", "VUS-low": "#8fb8ea"}
VUS_ONLY_ORDER = ["VUS-high", "VUS-mid", "VUS-low"]

GALL_COLORS = {"G+": "#2a9d55", "GVUS-high": "#1c5cab", "GVUS-mid": "#3b7dd8", "GVUS-low": "#8fb8ea", "G-": "#d9534f"}
GALL_ORDER = ["G+", "GVUS-high", "GVUS-mid", "GVUS-low", "G-"]
GALL_VUS_ONLY_ORDER = ["GVUS-high", "GVUS-mid", "GVUS-low"]

def pairwise_comparison(surv_df, group_col, group_a, group_b):
    pair = surv_df[surv_df[group_col].isin([group_a, group_b])].copy()
    pair["group_bin"] = (pair[group_col] == group_a).astype(int)
    a, b = pair[pair["group_bin"] == 1], pair[pair["group_bin"] == 0]
    lr = logrank_test(a["exit"], b["exit"], event_observed_A=a["event"], event_observed_B=b["event"])
    cph = CoxPHFitter()
    cph.fit(pair[["exit", "event", "group_bin"]], duration_col="exit", event_col="event")
    hr = np.exp(cph.params_["group_bin"])
    ci_lo, ci_hi = np.exp(cph.confidence_intervals_.loc["group_bin"])
    return hr, ci_lo, ci_hi, lr.p_value

def fit_all_groups(surv_df, group_col, order, colors):
    fitters = {}
    for group in order:
        g = surv_df[surv_df[group_col] == group]
        if len(g) == 0:
            continue
        kmf = KaplanMeierFitter(label=f"{group} (n={len(g)})")
        kmf.fit(durations=g["exit"], event_observed=g["event"])
        fitters[group] = kmf
    return fitters

def plot_reference_line_km(ax, fitters, colors, pos_group, neg_group, vus_order):
    for group, style in [(pos_group, "--"), (neg_group, ":")]:
        fitters[group].plot_survival_function(ax=ax, color=colors[group], ci_show=False, linewidth=1.6, linestyle=style)
    for group in vus_order:
        fitters[group].plot_survival_function(ax=ax, color=colors[group], ci_show=True, ci_alpha=0.15, linewidth=2.4)
    ax.set_xlim(0, 70); ax.set_ylim(0, 1)
    ax.set_xlabel("Age (years)"); ax.set_ylabel("Proportion Free of Overall Composite Endpoint")
    ax.legend(loc="lower left", fontsize=10)
    ax.grid(alpha=0.3)

def plot_forest(ax, surv_df, group_col, forest_order, colors, ref_group):
    forest_rows = []
    for group in forest_order[:-1]:
        hr, lo, hi, p = pairwise_comparison(surv_df, group_col, group, ref_group)
        forest_rows.append({"group": group, "hr": hr, "lo": lo, "hi": hi, "p": p, "is_ref": False})
    n_ref = surv_df[surv_df[group_col] == ref_group].shape[0]
    forest_rows.append({"group": ref_group, "hr": 1.0, "lo": None, "hi": None, "p": None, "is_ref": True, "n": n_ref})

    y_positions = list(range(len(forest_rows), 0, -1))
    for y, row in zip(y_positions, forest_rows):
        color = colors[row["group"]]
        if row["is_ref"]:
            ax.plot(1.0, y, marker="D", color=color, markersize=10, markeredgecolor="#333333", markeredgewidth=0.8)
            ax.text(1.35, y, f"1.00 (reference), n={row['n']}", va="center", fontsize=10, color="#333333")
        else:
            ax.errorbar(row["hr"], y, xerr=[[row["hr"] - row["lo"]], [row["hi"] - row["hr"]]],
                         fmt="o", color=color, ecolor=color, elinewidth=2, capsize=4, markersize=9)
            label = f"{row['hr']:.2f} ({row['lo']:.2f}-{row['hi']:.2f}), p={row['p']:.2g}"
            ax.text(row["hi"] * 1.08, y, label, va="center", fontsize=10, color="#333333")
    ax.axvline(1.0, color="#888888", linewidth=1.2, linestyle="--")
    ax.set_xscale("log")
    ax.set_xticks([0.5, 1, 1.5, 2, 3])
    ax.get_xaxis().set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())
    ax.set_yticks(y_positions)
    ax.set_yticklabels(forest_order, fontsize=11)
    ax.set_xlim(0.6, 3.5)
    ax.set_ylim(0.4, len(forest_rows) + 0.6)
    ax.set_xlabel(f"Hazard ratio vs {ref_group} (log scale)")
    ax.grid(axis="x", alpha=0.3)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    return forest_rows

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

def _label_panel(ax, letter):
    ax.text(-0.12, 1.04, letter, transform=ax.transAxes, fontsize=16, fontweight="bold", color="#0b0b0b")

# ---- Panel A: KM, 9 core sarcomere genes ----
fitters2 = fit_all_groups(surv2, "pipeline_group", GROUP2_ORDER, GROUP2_COLORS)
ax = axes[0, 0]
plot_reference_line_km(ax, fitters2, GROUP2_COLORS, "SARC+", "SARC-", VUS_ONLY_ORDER)
_label_panel(ax, "A")

# ---- Panel B: HR forest plot, 9 core sarcomere genes ----
ax = axes[0, 1]
plot_forest(ax, surv2, "pipeline_group", ["SARC+", "VUS-high", "VUS-mid", "VUS-low", "SARC-"], GROUP2_COLORS, "SARC-")
_label_panel(ax, "B")

# ---- Panel C: KM, all 29 HCM genes ----
fitters3 = fit_all_groups(surv3, "pipeline_group_all", GALL_ORDER, GALL_COLORS)
ax = axes[1, 0]
plot_reference_line_km(ax, fitters3, GALL_COLORS, "G+", "G-", GALL_VUS_ONLY_ORDER)
_label_panel(ax, "C")

# ---- Panel D: HR forest plot, all 29 HCM genes ----
ax = axes[1, 1]
plot_forest(ax, surv3, "pipeline_group_all", ["G+", "GVUS-high", "GVUS-mid", "GVUS-low", "G-"], GALL_COLORS, "G-")
_label_panel(ax, "D")

fig.tight_layout()
fig.savefig("outputs/hcm_share_4panel.png", dpi=300, facecolor="white")
print("saved outputs/hcm_share_4panel.png")
